In [1]:
import pandas as pd
import re
import json
import math
import os
from difflib import SequenceMatcher
from ollama import chat, ChatResponse
from tqdm import tqdm

In [2]:
# =========================
# Settings
# =========================
FILE_PATH = "./data/c3_m_test.xlsx"

# 定義三個模型變數 (可隨時修改)
M1 = "huihui_ai/internlm3-abliterated"
M2 = "glm4_9b"
M3 = "qwen3:8b"

# 第一階段：個別執行模型的清單
MODELS = [
    (M1, False), 
    (M2, False), 
    (M3, False)
]

# Our Proposed Method 設定
DEEPSEEK_THINK_MODEL = "qwen3:8b"
DEEPSEEK_THINK = True

TEMPERATURE = 0.0

# 自動生成輸出檔名 (處理冒號問題)
def get_out_filename(model_name):
    clean_name = model_name.replace("/", "_").replace(":", "_")
    return f"result_{clean_name}.xlsx"

OUT_MODEL_FILE = {
    "huihui_ai/internlm3-abliterated": "Results/m1/result_huihui_ai_internlm3-abliterated.xlsx",
    "glm4:9b": "Results/m1/result_glm4_9b.xlsx",
    "qwen3:8b": "Results/m1/result_qwen3_8b.xlsx",
}

# Ensemble 輸出檔
OUT_ENSEMBLE_FILE = "result_ensemble.xlsx"

# Accuracy summary 輸出檔
OUT_SUMMARY_XLSX = "accuracy_summary.xlsx"
OUT_SUMMARY_CSV = "accuracy_summary.csv"

In [3]:
# =========================
# Ollama call
# =========================
def query_ollama(prompt, model, CONTEXT_LEN=1536, think=False) -> str:
    try:
        resp: ChatResponse = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": CONTEXT_LEN, "temperature": TEMPERATURE},
            think=think
        )
        return resp.message.content
    except Exception as e:
        print(f"[Ollama Error] model={model} err={e}")
        return ""

In [4]:
# =========================
# Utility: safe cast / NaN handling
# =========================
def is_nan(x) -> bool:
    try:
        return x is not None and isinstance(x, float) and math.isnan(x)
    except Exception:
        return False

def to_int_1_4(x):
    if x is None or pd.isna(x):
        return None
    try:
        xi = int(x)
    except Exception:
        try:
            s = str(x).strip()
            if s in ["1", "2", "3", "4"]:
                return int(s)
            return None
        except Exception:
            return None
    return xi if xi in (1, 2, 3, 4) else None

def to_conf_0_1(x):
    if x is None or pd.isna(x):
        return 0.0
    try:
        xf = float(x)
    except Exception:
        return 0.0
    if math.isnan(xf):
        return 0.0
    return max(0.0, min(1.0, xf))

In [5]:
# =========================
# Parsing helpers
# =========================
def _strip_think(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def _normalize(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _best_option_match(text_cleaned: str, options: dict) -> int | None:
    t = _normalize(text_cleaned)
    if not t:
        return None
    for k, opt in options.items():
        o = _normalize(opt)
        if o and (o in t or t in o):
            return int(k)
    best_k, best_score = None, 0.0
    for k, opt in options.items():
        o = _normalize(opt)
        if not o:
            continue
        score = SequenceMatcher(None, t, o).ratio()
        if score > best_score:
            best_k, best_score = k, score
    if best_k is not None and best_score >= 0.55:
        return int(best_k)
    return None
    
def parse_answer_and_confidence(raw_text: str, options: dict) -> tuple[int | None, float | None]:
    if not raw_text:
        return None, None
    text = _strip_think(raw_text)

    # A) JSON
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            ans = to_int_1_4(obj.get("answer", None))
            conf = obj.get("confidence", None)
            conf = None if conf is None else float(conf)
            if conf is not None:
                conf = max(0.0, min(1.0, conf))
            if ans is not None:
                return ans, conf
        except Exception:
            pass

    # B) Answer / Confidence
    m1 = re.search(r"Answer:\s*([1-4])", text, re.IGNORECASE)
    m2 = re.search(r"Confidence:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    ans = int(m1.group(1)) if m1 else None
    conf = float(m2.group(1)) if m2 else None
    if conf is not None:
        conf = max(0.0, min(1.0, conf))
    if ans is not None:
        return ans, conf

    # C) fallback: last digit 1-4
    nums = re.findall(r"\b([1-4])\b", text)
    if nums:
        return int(nums[-1]), conf

    # D) options match
    matched = _best_option_match(text, options)
    return matched, conf

In [6]:
# =========================
# Prompt builder
# =========================
def build_prompt(article: str, question: str, valid_options: dict) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
    valid_numbers = ", ".join([str(k) for k in valid_options.keys()])
    return f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
1) Select the best option among ({valid_numbers}). If an option is missing/empty, do not select it.
2) Output ONLY a JSON object with:
   - "answer": an integer in [{valid_numbers}]
   - "confidence": a number in [0, 1] representing your probability the answer is correct.

Example:
{{"answer": 2, "confidence": 0.78}}
""".strip()

# =========================
# Ensemble helpers
# =========================
def majority_vote(preds: dict, best_model_name: str) -> int | None:
    counts = {}
    for a in preds.values():
        a = to_int_1_4(a)
        if a is None:
            continue
        counts[a] = counts.get(a, 0) + 1
    if not counts:
        return None
    best_a = max(counts, key=counts.get)
    top = [k for k, v in counts.items() if v == counts[best_a]]
    if len(top) > 1:
        return to_int_1_4(preds.get(best_model_name))
    return best_a
    
def confidence_weighted_vote(preds: dict, confs: dict, best_model_name: str) -> int | None:
    scores = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    total_mass = 0.0
    for m, a in preds.items():
        a = to_int_1_4(a)
        if a is None:
            continue
        w = to_conf_0_1(confs.get(m, None))
        scores[a] += w
        total_mass += w
    if total_mass <= 0.0:
        return None
    best_a = max(scores, key=lambda k: scores[k])
    best_score = scores[best_a]
    tied = [k for k, v in scores.items() if abs(v - best_score) < 1e-12]
    if len(tied) > 1:
        return to_int_1_4(preds.get(best_model_name))
    return best_a

In [7]:
# =========================
# Step 1: Run one model over whole dataset
# =========================
def run_single_model(df: pd.DataFrame, model_name: str, think: bool) -> tuple[str, int, int, float]:
    out_file = OUT_MODEL_FILE.get(model_name, f"result_{model_name.replace(':', '_')}.xlsx")
    
    # 【修改1】如果檔案已存在，直接讀取不重跑
    if os.path.exists(out_file):
        print(f"\n[Info] {model_name} 的輸出檔已存在 ({out_file})，直接讀取結果...")
        res_df = pd.read_excel(out_file)
        correct = int(res_df["Correct"].sum())
        total = len(res_df)
        acc = (correct / total * 100) if total else 0.0
        print(f"{model_name} Accuracy: {correct}/{total} = {acc:.2f}%")
        return out_file, correct, total, acc

    print(f"\n====== Running {model_name} (think={think}) ======")
    correct = 0
    results = []
    total = len(df)

    for _, row in tqdm(df.iterrows(), total=total, desc=f"Predicting[{model_name}]"):
        options_all = {1: row["Option1"], 2: row["Option2"], 3: row["Option3"], 4: row["Option4"]}
        valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}

        prompt = build_prompt(row["Article"], row["Question"], valid_options)
        raw = query_ollama(prompt, model=model_name, CONTEXT_LEN=1536, think=think)

        pred, conf = parse_answer_and_confidence(raw, valid_options)
        pred = to_int_1_4(pred)

        actual = to_int_1_4(row["Answer"])
        is_correct = (actual is not None and pred is not None and pred == actual)
        if is_correct:
            correct += 1

        results.append({
            "ID": row["ID"], "Actual": actual, "Predicted": pred,
            "Confidence": conf, "Correct": is_correct, "Model_Response": raw
        })

    acc = (correct / total * 100) if total else 0.0
    print(f"{model_name} Accuracy: {correct}/{total} = {acc:.2f}%")
    pd.DataFrame(results).to_excel(out_file, index=False)
    print(f"[Saved] {out_file}")
    return out_file, correct, total, acc

In [8]:
# =========================
# Step 2: Ensembles (Separated for clarity and timing)
# =========================
def run_ensembles_separately(df_base: pd.DataFrame, outputs: dict, best_model_name: str):
    df_g = pd.read_excel(outputs[M1])
    df_q = pd.read_excel(outputs[M2])
    df_d = pd.read_excel(outputs[M3])

    total = len(df_base)
    
    # 建立基礎的 DataFrame 來儲存所有 Ensemble 結果
    df_ensemble = pd.DataFrame()
    df_ensemble["ID"] = df_g["ID"]
    df_ensemble["Actual"] = df_g["Actual"]
    df_ensemble["Gemma_Pred"] = df_g["Predicted"]
    df_ensemble["Qwen_Pred"] = df_q["Predicted"]
    df_ensemble["DeepSeek_Pred"] = df_d["Predicted"]
    
    # 預備抓取用的 List
    actuals = df_g["Actual"].tolist()
    preds_g = df_g["Predicted"].tolist()
    preds_q = df_q["Predicted"].tolist()
    preds_d = df_d["Predicted"].tolist()
    confs_g = df_g["Confidence"].tolist()
    confs_q = df_q["Confidence"].tolist()
    confs_d = df_d["Confidence"].tolist()

    # 1. Ensemble - Majority Vote
    maj_correct = 0
    maj_preds, maj_corrs = [], []
    for i in tqdm(range(total), desc="Running [Majority Voting (MV)]"):
        preds = {M1: preds_g[i], M2: preds_q[i], M3: preds_d[i]}
        ans = majority_vote(preds, best_model_name)
        is_corr = (ans is not None and ans == actuals[i])
        maj_preds.append(ans)
        maj_corrs.append(is_corr)
        if is_corr: maj_correct += 1
        
    df_ensemble["MajorityVote"] = maj_preds
    df_ensemble["MajorityVote_Correct"] = maj_corrs
    maj_acc = (maj_correct / total * 100) if total else 0.0
    print(f"Majority Accuracy: {maj_correct}/{total} = {maj_acc:.2f}%")

    # 2. Ensemble - Confidence Weighted Vote
    wv_correct = 0
    wv_preds, wv_corrs = [], []
    for i in tqdm(range(total), desc="Running [Confidence-Weighted Voting (CWV)]"):
        preds = {M1: preds_g[i], M2: preds_q[i], M3: preds_d[i]}
        confs = {M1: confs_g[i], M2: confs_q[i], M3: confs_d[i]}
        ans = confidence_weighted_vote(preds, confs, best_model_name)
        is_corr = (ans is not None and ans == actuals[i])
        wv_preds.append(ans)
        wv_corrs.append(is_corr)
        if is_corr: wv_correct += 1

    df_ensemble["Confidence_WeightedVote"] = wv_preds
    df_ensemble["Confidence_WeightedVote_Correct"] = wv_corrs
    wv_acc = (wv_correct / total * 100) if total else 0.0
    print(f"Weighted Accuracy: {wv_correct}/{total} = {wv_acc:.2f}%")

    # 3. Ours - Cascaded Conflict-Resolution Ensemble
    ours_correct = 0
    r6_preds, r6_corrs, r6_disagrees = [], [], []
    ds_think_preds, ds_think_resps = [], []
    
    # 統計用變數
    disagree_count = 0
    think_none_count, think_not_none_count = 0, 0
    think_none_correct, think_not_none_correct = 0, 0

    for i in tqdm(range(total), desc="Running [Our Proposed CCRE Method]"):
        row = df_base.iloc[i]
        actual = actuals[i]
        
        g_ans = to_int_1_4(preds_g[i])
        q_ans = to_int_1_4(preds_q[i])
        
        # 決定 Best Model 答案 (對應迴圈當前 index)
        if best_model_name == M1:
            best_ans = g_ans
        elif best_model_name == M2:
            best_ans = q_ans
        else:
            best_ans = to_int_1_4(preds_d[i])

        is_disagree = True
        if g_ans is not None and g_ans == q_ans:
            is_disagree = False
            
        r6_disagrees.append(is_disagree)

        if not is_disagree:
            ours_pred = g_ans
            ds_think_preds.append(None)
            ds_think_resps.append("")
        else:
            disagree_count += 1
            
            # 組裝提示詞，呼叫 Thinking Model
            options_all = {1: row["Option1"], 2: row["Option2"], 3: row["Option3"], 4: row["Option4"]}
            valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}
            prompt = build_prompt(row["Article"], row["Question"], valid_options)
            
            ds_raw = query_ollama(prompt, model=DEEPSEEK_THINK_MODEL, CONTEXT_LEN=3192, think=DEEPSEEK_THINK)
            ds_pred, _ = parse_answer_and_confidence(ds_raw, valid_options)
            ds_pred = to_int_1_4(ds_pred)
            
            ds_think_preds.append(ds_pred)
            ds_think_resps.append(ds_raw)

            # 【修改3】當 DeepSeek-Think 為 None 時，使用 best_ans 而非寫死 qwen_ans
            if ds_pred is None:
                think_none_count += 1
                ours_pred = best_ans
                if ours_pred is not None and ours_pred == actual:
                    think_none_correct += 1
            else:
                think_not_none_count += 1
                ours_pred = ds_pred
                if ours_pred is not None and ours_pred == actual:
                    think_not_none_correct += 1

        r6_preds.append(ours_pred)
        is_corr = (ours_pred is not None and ours_pred == actual)
        r6_corrs.append(is_corr)
        if is_corr: ours_correct += 1

    df_ensemble["Ours_Disagreement"] = r6_disagrees
    df_ensemble["Ours_Final"] = r6_preds
    df_ensemble["Ours_Correct"] = r6_corrs
    df_ensemble["DeepSeekThink_Pred"] = ds_think_preds
    df_ensemble["DeepSeekThink_Response"] = ds_think_resps

    r6_acc = (ours_correct / total * 100) if total else 0.0
    print(f"Our Proposed Method Accuracy: {ours_correct}/{total} = {r6_acc:.2f}%")

    # 【修改4】印出詳細的統計數據
    print("\n" + "=" * 40)
    print("====== Our Proposed Method 詳細統計 ======")
    print(f"總數據筆數: {total}")
    disagree_pct = (disagree_count / total * 100) if total else 0
    print(f"M1 與 M2 不一致筆數 (觸發 Thinking Model): {disagree_count} 筆 ({disagree_pct:.2f}%)")
    
    if disagree_count > 0:
        tn_pct = (think_none_count / disagree_count * 100)
        print(f" ├─ Thinking Model 回答為 None (採用 Best Model): {think_none_count} 筆 ({tn_pct:.2f}%)")
        if think_none_count > 0:
            tnc_pct = (think_none_correct / think_none_count * 100)
            print(f" │   └─ Best Model 救援正確: {think_none_correct} 筆 ({tnc_pct:.2f}%)")
            
        tnn_pct = (think_not_none_count / disagree_count * 100)
        print(f" └─ Thinking Model 成功給出作答: {think_not_none_count} 筆 ({tnn_pct:.2f}%)")
        if think_not_none_count > 0:
            tnnc_pct = (think_not_none_correct / think_not_none_count * 100)
            print(f"     └─ Thinking Model 作答正確: {think_not_none_correct} 筆 ({tnnc_pct:.2f}%)")
    print("=" * 40 + "\n")

    df_ensemble.to_excel(OUT_ENSEMBLE_FILE, index=False)
    print(f"[Saved] {OUT_ENSEMBLE_FILE}")

    return (maj_correct, maj_acc), (wv_correct, wv_acc), (ours_correct, r6_acc)

In [9]:
# =========================
# Entry
# =========================
print(f"Loading dataset: {FILE_PATH}")
df = pd.read_excel(FILE_PATH)

required_columns = ["ID", "Article", "Question", "Option1", "Option2", "Option3", "Option4", "Answer"]
if not all(c in df.columns for c in required_columns):
    raise ValueError(f"Excel 缺少必要欄位，需要: {required_columns}")

Loading dataset: ./data/c3_m_test.xlsx


In [10]:
outputs = {}
summary_rows = []
model_accuracies = {}

# 1) 各模型跑完整 dataset (會自動偵測檔案略過)
for model_name, think in MODELS:
    out_file, correct, total, acc = run_single_model(df, model_name, think)
    outputs[model_name] = out_file
    model_accuracies[model_name] = acc
    summary_rows.append({
        "Method": model_name,
        "Correct": correct,
        "Total": total,
        "Accuracy(%)": round(acc, 4)
    })

# 找出準確率最高的模型
best_model_name = max(model_accuracies, key=model_accuracies.get)
print(f"\n[Info] 整體準確率最高的模型為: {best_model_name} (Accuracy: {model_accuracies[best_model_name]:.2f}%)")


[Info] huihui_ai/internlm3-abliterated 的輸出檔已存在 (Results/m1/result_huihui_ai_internlm3-abliterated.xlsx)，直接讀取結果...
huihui_ai/internlm3-abliterated Accuracy: 1884/2002 = 94.11%

[Info] glm4_9b 的輸出檔已存在 (result_glm4_9b.xlsx)，直接讀取結果...
glm4_9b Accuracy: 1871/2002 = 93.46%

[Info] qwen3:8b 的輸出檔已存在 (Results/m1/result_qwen3_8b.xlsx)，直接讀取結果...
qwen3:8b Accuracy: 1869/2002 = 93.36%

[Info] 整體準確率最高的模型為: huihui_ai/internlm3-abliterated (Accuracy: 94.11%)


In [11]:
print("\n開始進行 Ensemble 計算...\n")

# 2) 執行 Ensemble 策略 (會分開顯示進度條)
(maj_correct, maj_acc), (wv_correct, wv_acc), (r6_correct, r6_acc) = run_ensembles_separately(
    df_base=df,
    outputs=outputs,
    best_model_name=best_model_name
)

summary_rows.extend([
    {"Method": "Ensemble-Majority (3 models)", "Correct": maj_correct, "Total": len(df), "Accuracy(%)": round(maj_acc, 4)},
    {"Method": "Ensemble-Confidence_Weighted (3 models)", "Correct": wv_correct, "Total": len(df), "Accuracy(%)": round(wv_acc, 4)},
    {"Method": "Our Proposed Method", "Correct": r6_correct, "Total": len(df), "Accuracy(%)": round(r6_acc, 4)},
])

summary_df = pd.DataFrame(summary_rows).sort_values(by="Accuracy(%)", ascending=False)
summary_df.to_excel(OUT_SUMMARY_XLSX, index=False)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"\n[Saved] {OUT_SUMMARY_XLSX}")
print(f"[Saved] {OUT_SUMMARY_CSV}")
print("\n===== Accuracy Summary =====")
print(summary_df.to_string(index=False))


開始進行 Ensemble 計算...



Running [Majority Voting (MV)]: 100%|██████████████████████████████████████████| 2002/2002 [00:00<00:00, 199463.08it/s]


Majority Accuracy: 1893/2002 = 94.56%


Running [Confidence-Weighted Voting (CWV)]: 100%|███████████████████████████████| 2002/2002 [00:00<00:00, 97782.76it/s]


Weighted Accuracy: 1893/2002 = 94.56%


Running [Our Proposed CCRE Method]: 100%|████████████████████████████████████████| 2002/2002 [3:47:02<00:00,  6.80s/it]


Our Proposed Method Accuracy: 1909/2002 = 95.35%

====== Our Proposed Method 詳細統計 ======
總數據筆數: 2002
M1 與 M2 不一致筆數 (觸發 Thinking Model): 120 筆 (5.99%)
 ├─ Thinking Model 回答為 None (採用 Best Model): 9 筆 (7.50%)
 │   └─ Best Model 救援正確: 6 筆 (66.67%)
 └─ Thinking Model 成功給出作答: 111 筆 (92.50%)
     └─ Thinking Model 作答正確: 82 筆 (73.87%)

[Saved] result_ensemble.xlsx

[Saved] accuracy_summary.xlsx
[Saved] accuracy_summary.csv

===== Accuracy Summary =====
                                 Method  Correct  Total  Accuracy(%)
                    Our Proposed Method     1909   2002      95.3546
           Ensemble-Majority (3 models)     1893   2002      94.5554
Ensemble-Confidence_Weighted (3 models)     1893   2002      94.5554
        huihui_ai/internlm3-abliterated     1884   2002      94.1059
                                glm4_9b     1871   2002      93.4565
                               qwen3:8b     1869   2002      93.3566
